# Cosmos Client

This notebook is the nbdev source for the WellSpot read-only Cosmos Gremlin client.

Caller-managed values such as Key Vault URL, environment mappings, and any secret-name overrides stay outside this notebook.

In [ ]:
#| default_exp client

In [ ]:
#| export
from __future__ import annotations

import json
import time
from contextlib import contextmanager
from dataclasses import dataclass
from typing import Any, Optional, Tuple

import pandas as pd
from azure.cosmos import CosmosClient
from azure.identity import ClientSecretCredential, DefaultAzureCredential
from azure.keyvault.secrets import SecretClient
from gremlin_python.driver.client import Client
from gremlin_python.driver.serializer import GraphSONSerializersV2d0

In [ ]:
#| export
@dataclass(frozen=True)
class SecretNames:
    """Centralized secret names used by the read-only client."""

    tenant_id: str = "AZURE-TENANT-ID"
    client_id: str = "GRAPH-CLIENT-ID"
    client_secret: str = "GRAPH-CLIENT-SECRET"
    gremlin_url: str = "GREMLIN-URL"
    cosmos_url: str = "COSMOS-URL"


@dataclass(frozen=True)
class CosmosEnvironmentConfig:
    """Container coordinates for a specific Cosmos environment."""

    database_name: str
    container_name: str

In [ ]:
#| export
class SecretClientManager:
    def __init__(
        self,
        key_vault_url: str,
        credential: DefaultAzureCredential | None = None,
    ) -> None:
        self.key_vault_url = key_vault_url
        self._credential = credential
        self._secret_client: SecretClient | None = None

    def get_credential(self) -> DefaultAzureCredential:
        if self._credential is None:
            self._credential = DefaultAzureCredential()
        return self._credential

    def get_secret_client(self) -> SecretClient:
        if self._secret_client is None:
            self._secret_client = SecretClient(
                vault_url=self.key_vault_url,
                credential=self.get_credential(),
            )
        return self._secret_client

    def get_secret_value(self, secret_name: str) -> str:
        return self.get_secret_client().get_secret(secret_name).value


class GremlinClientManager:
    def __init__(
        self,
        environment: CosmosEnvironmentConfig,
        secret_client_manager: SecretClientManager,
        secret_names: SecretNames | None = None,
    ) -> None:
        self.environment = environment
        self.secret_client_manager = secret_client_manager
        self.secret_names = secret_names or SecretNames()
        self.credential = ClientSecretCredential(
            self.secret_client_manager.get_secret_value(self.secret_names.tenant_id),
            self.secret_client_manager.get_secret_value(self.secret_names.client_id),
            self.secret_client_manager.get_secret_value(self.secret_names.client_secret),
        )
        self.token: str | None = None
        self.token_expiration = 0.0
        self.gremlin_endpoint = self.secret_client_manager.get_secret_value(self.secret_names.gremlin_url)
        self.endpoint = self.secret_client_manager.get_secret_value(self.secret_names.cosmos_url)
        self.get_token()

    def get_token(self) -> None:
        token_response = self.credential.get_token("https://cosmos.azure.com/.default")
        self.token = token_response.token
        self.token_expiration = float(token_response.expires_on)

    def ensure_token_validity(self) -> None:
        if time.time() >= self.token_expiration - 300:
            self.get_token()

    @contextmanager
    def client(self):
        self.ensure_token_validity()
        gremlin_client = Client(
            url=self.gremlin_endpoint,
            traversal_source="g",
            username=(
                f"/dbs/{self.environment.database_name}/colls/"
                f"{self.environment.container_name}"
            ),
            password=self.token,
            message_serializer=GraphSONSerializersV2d0(),
        )

        try:
            yield gremlin_client
        finally:
            gremlin_client.close()

    def get_client_cosmos(self) -> CosmosClient:
        return CosmosClient(self.endpoint, self.credential)

    def get_database(self):
        return self.get_client_cosmos().get_database_client(self.environment.database_name)

    def get_container(self):
        return self.get_database().get_container_client(self.environment.container_name)

In [ ]:
#| export
class CosmosGraphClient:
    """Read-only client over Cosmos Gremlin."""

    def __init__(self, gremlin_manager: GremlinClientManager, df_response: bool = True, debug: bool = False):
        self._gm = gremlin_manager
        self.df_response = df_response
        self.debug = debug
        self.endpoints = self._Endpoints(self)

    def _submit(self, gremlin: str, bindings: Optional[dict] = None):
        if self.debug:
            print("Gremlin:", gremlin)
            if bindings:
                print("Bindings:", bindings)
        with self._gm.client() as gremlin_client:
            result = gremlin_client.submit(gremlin, bindings=bindings or {})
            return result.all().result()

    @staticmethod
    def _flatten_graphson_item(item: Any) -> dict:
        if isinstance(item, dict) and "type" in item:
            item_type = item.get("type")
            if item_type == "vertex":
                row = {"id": item.get("id"), "label": item.get("label")}
                properties = item.get("properties", {}) or {}
                for key, values in properties.items():
                    if isinstance(values, list) and values:
                        row[key] = (
                            values[0].get("value")
                            if len(values) == 1
                            else [value.get("value") for value in values]
                        )
                    else:
                        row[key] = values
                return row
            if item_type == "edge":
                row = {
                    "id": item.get("id"),
                    "label": item.get("label"),
                    "inV": item.get("inV"),
                    "outV": item.get("outV"),
                }
                properties = item.get("properties", {}) or {}
                for key, value in properties.items():
                    row[key] = value
                return row
        if isinstance(item, dict):
            row = {}
            for key, value in item.items():
                if isinstance(value, list) and len(value) == 1:
                    row[key] = value[0]
                else:
                    row[key] = value
            return row
        return {"value": item}

    def _to_frame(self, data: Any, *, request_id=None, request_id_col: str = "request_id") -> pd.DataFrame:
        if isinstance(data, list):
            rows = [self._flatten_graphson_item(item) for item in data]
            frame = pd.DataFrame(rows)
        elif isinstance(data, dict):
            frame = pd.DataFrame([self._flatten_graphson_item(data)])
        else:
            frame = pd.DataFrame({"value": [data]})
        if request_id is not None:
            frame[request_id_col] = request_id
        return frame

    @staticmethod
    def _sid_pair(analysis_id: Any) -> Tuple[str, Optional[int]]:
        string_value = str(analysis_id).strip()
        try:
            numeric_value = int(string_value)
        except Exception:
            numeric_value = None
        return string_value, numeric_value

    class _Endpoints:
        def __init__(self, client: "CosmosGraphClient"):
            self._client = client

        def _maybe_df(self, data, as_df: Optional[bool], request_id=None, request_id_col: str = "request_id"):
            to_df = self._client.df_response if as_df is None else as_df
            if to_df:
                return self._client._to_frame(data, request_id=request_id, request_id_col=request_id_col)
            return data

        def get_info(self, *, as_df: Optional[bool] = None):
            query = "g.V().label().groupCount()"
            result = self._client._submit(query)
            return self._maybe_df(result[0] if result else {}, as_df)

        def get_all_h4(self, *, as_df: Optional[bool] = None):
            query = "g.V().hasLabel('h4').valueMap(true)"
            result = self._client._submit(query)
            return self._maybe_df(result, as_df)

        def get_h4_status_by_analysisId(
            self,
            analysisId: int,
            *,
            as_df: Optional[bool] = None,
            request_id_col: str = "request_id",
        ):
            sid_s, sid_n = self._client._sid_pair(analysisId)
            if sid_n is not None:
                query = (
                    "g.V().hasLabel('analysis_status')"
                    ".or(__.has('request_id', sid_s), __.has('request_id', sid_n))"
                    ".valueMap(true)"
                )
                bindings = {"sid_s": sid_s, "sid_n": sid_n}
            else:
                query = (
                    "g.V().hasLabel('h4_status')"
                    ".has('request_id', sid_s)"
                    ".valueMap(true)"
                )
                bindings = {"sid_s": sid_s}
            result = self._client._submit(query, bindings=bindings)
            return self._maybe_df(result, as_df, request_id=analysisId, request_id_col=request_id_col)

        def get_h4_op_history_by_analysisId(
            self,
            analysisId: int,
            *,
            as_df: Optional[bool] = None,
            request_id_col: str = "request_id",
        ):
            sid_s, sid_n = self._client._sid_pair(analysisId)
            if sid_n is not None:
                query = (
                    "g.V().hasLabel('h4_operation')"
                    ".or(__.has('request_id', sid_s), __.has('request_id', sid_n))"
                    ".order().by(coalesce(values('startDay'), constant(0)))"
                    ".valueMap(true)"
                )
                bindings = {"sid_s": sid_s, "sid_n": sid_n}
            else:
                query = (
                    "g.V().hasLabel('h4_operation')"
                    ".has('request_id', sid_s)"
                    ".order().by(coalesce(values('startDay'), constant(0)))"
                    ".valueMap(true)"
                )
                bindings = {"sid_s": sid_s}
            result = self._client._submit(query, bindings=bindings)
            return self._maybe_df(result, as_df, request_id=analysisId, request_id_col=request_id_col)

        def get_h4_op_history_timelines_by_analysisId(
            self,
            analysisId: int,
            *,
            as_df: Optional[bool] = None,
            request_id_col: str = "request_id",
        ):
            sid_s, sid_n = self._client._sid_pair(analysisId)
            if sid_n is not None:
                query = (
                    "g.V().hasLabel('h4_timelines')"
                    ".or(__.has('request_id', sid_s), __.has('request_id', sid_n))"
                    ".valueMap(true)"
                )
                bindings = {"sid_s": sid_s, "sid_n": sid_n}
            else:
                query = (
                    "g.V().hasLabel('h4_timelines')"
                    ".has('request_id', sid_s)"
                    ".valueMap(true)"
                )
                bindings = {"sid_s": sid_s}
            result = self._client._submit(query, bindings=bindings)

            to_df = self._client.df_response if as_df is None else as_df
            if not to_df:
                if result:
                    value_map = self._client._flatten_graphson_item(result[0])
                    timelines_json = value_map.get("timelinesJson")
                    try:
                        return json.loads(timelines_json) if isinstance(timelines_json, str) else []
                    except Exception:
                        return []
                return []

            if result:
                value_map = self._client._flatten_graphson_item(result[0])
                timelines_json = value_map.get("timelinesJson")
                try:
                    timelines = json.loads(timelines_json) if isinstance(timelines_json, str) else []
                except Exception:
                    timelines = []
            else:
                timelines = []
            return pd.DataFrame([{"timelines": timelines, request_id_col: int(analysisId)}])

        def get_h4_fatigue_by_analysisId(
            self,
            analysisId: int,
            *,
            as_df: Optional[bool] = None,
            request_id_col: str = "request_id",
        ):
            sid_s, sid_n = self._client._sid_pair(analysisId)
            if sid_n is not None:
                query = (
                    "g.V().hasLabel('results')"
                    ".or(__.has('request_id', sid_s), __.has('request_id', sid_n))"
                    ".valueMap(true)"
                )
                bindings = {"sid_s": sid_s, "sid_n": sid_n}
            else:
                query = (
                    "g.V().hasLabel('h4_results')"
                    ".has('request_id', sid_s)"
                    ".valueMap(true)"
                )
                bindings = {"sid_s": sid_s}
            result = self._client._submit(query, bindings=bindings)

            payload = None
            if result:
                value_map = self._client._flatten_graphson_item(result[0])
                if "resultsJson" in value_map and isinstance(value_map["resultsJson"], str):
                    try:
                        payload = json.loads(value_map["resultsJson"])
                    except Exception:
                        payload = None

            to_df = self._client.df_response if as_df is None else as_df
            if not to_df:
                return payload if payload is not None else (result[0] if result else {})

            if payload is not None:
                frame = pd.json_normalize(payload)
                frame[request_id_col] = int(analysisId)
                return frame
            return self._client._to_frame(result, request_id=int(analysisId), request_id_col=request_id_col)

        def get_h4_by_member_or_analysisId(
            self,
            member_or_id: str,
            *,
            as_df: Optional[bool] = None,
            request_id_col: str = "request_id",
        ):
            sid_s = str(member_or_id).strip()
            try:
                sid_n = int(sid_s)
            except Exception:
                sid_n = None

            if sid_n is not None:
                query = (
                    "g.V().hasLabel('h4')"
                    ".or(__.has('source_id', sid_n), __.has('unique-id', sid_s))"
                    ".valueMap(true)"
                )
                bindings = {"sid_s": sid_s, "sid_n": sid_n}
            else:
                query = (
                    "g.V().hasLabel('h4')"
                    ".has('unique-id', sid_s)"
                    ".valueMap(true)"
                )
                bindings = {"sid_s": sid_s}

            result = self._client._submit(query, bindings=bindings)
            return self._maybe_df(result, as_df, request_id=member_or_id, request_id_col=request_id_col)

In [ ]:
EXAMPLE_ENVIRONMENT = {
    "database_name": "<database-name>",
    "container_name": "<container-name>",
}

EXAMPLE_ENVIRONMENT